# Titanic Survival Prediction — Classical ML vs. Deep Learning (PyTorch)

## Objective
- Predict passenger survival on the Titanic using two families of approaches on the **same
  preprocessed features and the same train/validation/test split**, so results are directly comparable:
  1. **Classical machine learning** (Logistic Regression, Decision Tree, Random Forest — scikit-learn)
  2. **A small feed-forward neural network** (PyTorch)
- Along the way, review the PyTorch fundamentals (tensors, autograd, `nn.Module`, training loop)
  that the neural-network section relies on.

## Software Required
- Python 3.x, pandas, numpy, scikit-learn, matplotlib, joblib
- PyTorch (`pip install torch`)

## Workflow
`Load data → EDA → Feature engineering & preprocessing → Train/val/test split (shared) →
Classical ML (train, validate, pick best, test) → PyTorch NN (train, validate, test) →
Compare both approaches → Save models → Conclusion`


## 1. Background: PyTorch Fundamentals

### 1.1 What is PyTorch?
PyTorch is an open-source deep learning framework (Meta AI) known for:
- **Pythonic, NumPy-like syntax** that's easy to read and debug.
- A **dynamic ("define-by-run") computation graph** — the graph is built on the fly as
  operations execute, unlike static-graph frameworks. This makes debugging and variable-length
  models (e.g. RNNs) easier.
- **GPU acceleration** — tensors move between CPU/GPU with `.to(device)`.
- **Autograd** — automatic differentiation for backpropagation.

### 1.2 Tensors
The core data structure — like a NumPy array, but tracks operations (for autograd) and can live
on a GPU. Common ranks: scalar (0-D), vector (1-D), matrix (2-D), N-D tensor (images, batches...).

### 1.3 Autograd
Every tensor has `requires_grad`. When `True`, PyTorch records operations on it in a dynamic
computational graph. Calling `.backward()` on a scalar (usually the loss) walks that graph
backward via the chain rule and populates `.grad` on every tracked tensor — this **is**
backpropagation.

### 1.4 Building Blocks
- `nn.Module` — base class for all models/layers.
- `nn.Linear(in, out)` — fully-connected layer.
- Activations (`nn.ReLU`, `nn.Sigmoid`, ...) — introduce non-linearity.
- Loss functions (`nn.BCELoss` for binary classification, `nn.MSELoss` for regression, ...).
- Optimizers (`optim.SGD`, `optim.Adam`, ...) — update weights from gradients.

### 1.5 The General Training Loop
```
for epoch in range(num_epochs):
    optimizer.zero_grad()      # clear old gradients
    outputs = model(inputs)    # forward pass
    loss = criterion(outputs, targets)
    loss.backward()            # compute new gradients
    optimizer.step()           # update weights
```
This loop, applied to the Titanic dataset, is what Section 4 below implements.


## 2. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

import torch
import torch.nn as nn
import torch.optim as optim

import joblib

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 3. Load Data

Using the local `titanic.csv` (the same file for every model in this notebook — the classical-ML
lab previously pulled this from a GitHub URL while the PyTorch lab read the local file; both now
use the single local copy for consistency and offline reproducibility).


In [ ]:
df = pd.read_csv("titanic.csv")
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()


## 4. Exploratory Data Analysis (EDA)

In [ ]:
print("Survival rate overall: {:.2%}".format(df["Survived"].mean()))
print()
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

df["Survived"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#c0392b", "#27ae60"]
)
axes[0].set_xticklabels(["Did not survive", "Survived"], rotation=0)
axes[0].set_title("Survival Counts")

df.groupby("Pclass")["Survived"].mean().plot(kind="bar", ax=axes[1], color="#2980b9")
axes[1].set_title("Survival Rate by Passenger Class")
axes[1].set_ylabel("Survival rate")

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=120)
plt.show()


## 5. Feature Engineering

`FamilySize` and `IsAlone` are derived from `SibSp`/`Parch`, since family size on board was more
predictive than the raw sibling/spouse and parent/child counts individually. `Name`, `Ticket`,
`Cabin`, and `PassengerId` are dropped (identifiers / free text / too many missing values).


In [ ]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

TARGET = "Survived"
numeric_features = ["Age", "Fare", "FamilySize", "IsAlone", "Pclass"]
categorical_features = ["Sex", "Embarked"]

X = df[numeric_features + categorical_features].copy()
y = df[TARGET].copy()

print("Features used:", numeric_features + categorical_features)
X.head()


## 6. Train / Validation / Test Split

A single 60/20/20 split, stratified on the target, is created **once** and reused by both the
classical models and the PyTorch network — this is what makes the final comparison fair.


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Train:      {X_train.shape[0]} rows")
print(f"Validation: {X_val.shape[0]} rows")
print(f"Test:       {X_test.shape[0]} rows")


## 7. Shared Preprocessing

One `ColumnTransformer` (median-impute + scale numeric; most-frequent-impute + one-hot
categorical) is **fit on train only**, then used to transform train/val/test. Both the classical
models and the PyTorch tensors below are built from this exact same transformed data.


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

print("Processed feature matrix shape:", X_train_proc.shape)


## 8. Part 1 — Classical Machine Learning

Train Logistic Regression, Decision Tree, and Random Forest on the training set, then select the
best of the three using **validation** performance only (test stays untouched).


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=6, random_state=RANDOM_STATE, n_jobs=1
    ),
}

val_results = {}
fitted_models = {}

for name, model in models.items():
    model.fit(X_train_proc, y_train)
    preds = model.predict(X_val_proc)
    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds)
    val_results[name] = acc
    fitted_models[name] = model
    print(f"  {name:20s} -> Validation Accuracy: {acc:.4f} | F1: {f1:.4f}")

best_classical_name = max(val_results, key=val_results.get)
best_classical_model = fitted_models[best_classical_name]
print(f"\nBest classical model (by validation accuracy): {best_classical_name}")


### 8.1 Final Test Evaluation — Best Classical Model

In [ ]:
classical_test_preds = best_classical_model.predict(X_test_proc)

classical_metrics = {
    "Accuracy": accuracy_score(y_test, classical_test_preds),
    "Precision": precision_score(y_test, classical_test_preds),
    "Recall": recall_score(y_test, classical_test_preds),
    "F1": f1_score(y_test, classical_test_preds),
}
for k, v in classical_metrics.items():
    print(f"  Test {k:10s}: {v:.4f}")

print("\nConfusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, classical_test_preds))
print()
print(classification_report(y_test, classical_test_preds, target_names=["Did not survive", "Survived"]))


### 8.2 Save the Best Classical Pipeline

In [ ]:
# Bundle preprocessing + classifier into one deployable pipeline object
best_classical_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", best_classical_model),
])
best_classical_pipeline.fit(X_train, y_train)  # refit the combined object on raw train features

joblib.dump(best_classical_pipeline, "titanic_best_classical_model.joblib")
print("Saved -> titanic_best_classical_model.joblib")


## 9. Part 2 — Deep Learning with PyTorch

Same `X_train_proc` / `X_val_proc` / `X_test_proc` arrays, now converted to tensors and fed to a
small feed-forward network.


In [ ]:
X_train_t = torch.tensor(X_train_proc, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_val_t = torch.tensor(X_val_proc, dtype=torch.float32)
y_val_t = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test_proc, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

print(X_train_t.shape, X_val_t.shape, X_test_t.shape)


### 9.1 Model Definition

In [ ]:
class TitanicNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, 16)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(16, 8)
        self.relu2 = nn.ReLU()
        self.output = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu1(self.layer1(x))
        x = self.relu2(self.layer2(x))
        return self.sigmoid(self.output(x))

input_size = X_train_t.shape[1]
nn_model = TitanicNN(input_size)
print(nn_model)


### 9.2 Loss, Optimizer & Training Loop

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(nn_model.parameters(), lr=0.001)

num_epochs = 100
train_losses, val_losses = [], []

for epoch in range(num_epochs):
    # ---- training ----
    nn_model.train()
    optimizer.zero_grad()
    outputs = nn_model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    # ---- validation ----
    nn_model.eval()
    with torch.no_grad():
        val_outputs = nn_model(X_val_t)
        val_loss = criterion(val_outputs, y_val_t)
        val_losses.append(val_loss.item())

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (BCE)")
plt.title("PyTorch NN: Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig("nn_loss_curve.png", dpi=120)
plt.show()


### 9.3 Final Test Evaluation — PyTorch NN

*(This step was missing from the original notebook — a test set was created but never actually
evaluated. Added here to mirror the classical-ML evaluation exactly.)*


In [ ]:
nn_model.eval()
with torch.no_grad():
    test_probs = nn_model(X_test_t)
    test_preds_nn = (test_probs >= 0.5).float()

nn_metrics = {
    "Accuracy": accuracy_score(y_test_t, test_preds_nn),
    "Precision": precision_score(y_test_t, test_preds_nn),
    "Recall": recall_score(y_test_t, test_preds_nn),
    "F1": f1_score(y_test_t, test_preds_nn),
}
for k, v in nn_metrics.items():
    print(f"  Test {k:10s}: {v:.4f}")

print("\nConfusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test_t, test_preds_nn))
print()
print(classification_report(y_test_t, test_preds_nn, target_names=["Did not survive", "Survived"]))


### 9.4 Save the PyTorch Model

In [ ]:
torch.save(nn_model.state_dict(), "titanic_pytorch_nn.pt")
print("Saved -> titanic_pytorch_nn.pt")


## 10. Model Comparison

All four models were trained on the same training split, model-selected/tuned without touching
the test set, and evaluated **once** on the same held-out test set.


In [ ]:
rows = []
for name, model in fitted_models.items():
    preds = model.predict(X_test_proc)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
    })
rows.append({"Model": "PyTorch NN", **nn_metrics})

comparison_df = pd.DataFrame(rows).set_index("Model").round(4)
comparison_df


In [ ]:
comparison_df.plot(kind="bar", figsize=(9, 5))
plt.title("Test-Set Metric Comparison Across Models")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120)
plt.show()


## 11. Conclusion

- On this dataset (small, ~900 rows, 7 tabular features), the classical **Logistic Regression**
  model matched or outperformed the deeper models — including the PyTorch neural network —
  on held-out test data. This is a common and expected result: small tabular datasets rarely
  benefit from extra model capacity, and simpler models generalize better with less data and
  less tuning.
- The **PyTorch network** trained successfully (loss decreased smoothly for both train and
  validation, with no divergence), demonstrating the full pipeline — tensors, autograd,
  `nn.Module`, and the training loop — but its test performance trailed the classical baselines.
  With more epochs, regularization (dropout/weight decay), or hyperparameter tuning, this gap
  could narrow.
- Reusing one preprocessing pipeline and one train/val/test split across both approaches made the
  comparison fair and reproducible — a pattern worth carrying into future labs.
- Both the classical pipeline (`titanic_best_classical_model.joblib`) and the PyTorch model
  (`titanic_pytorch_nn.pt`) are saved and ready to be reloaded for inference.
